# Подготовка отправки на Codabench

1. Обучаем выбранную модель на **всех** публичных данных (train + val + test, все 10 участников): для отправки больше данных лучше.
2. Экспортируем её в папку и ZIP: `submission.py` (самодостаточный файл из `submission/`) + `model.npz` (только массивы numpy, читается любой версией numpy).
3. Проверяем, что модель из ZIP предсказывает ровно то же, что исходная.
4. Загружаем ZIP в **My Submissions**.

Поддерживаются `logreg` и `forest`. Помните про лимит: в финальном этапе 1 отправка в день и 5 всего.

То же одной командой из папки `track2_bci`: `python -m bci.make_submission` (обучение на всех данных, ZIP и все проверки).


In [ ]:
import sys; sys.path.append("..")
from bci.data import load_split, concat
from bci.evaluation import predict_streams
from bci.export import save_submission, load_submission
from bci.models import logreg, forest

train, val, test = (load_split(s, "../data") for s in ["train", "val", "test"])
full = concat(train, val, test)

## Обучение
Параметры возьмите из ноутбука модели — те, что лучше всего на кросс-валидации.

In [ ]:
NAME, MODEL = "forest", forest
PARAMS = dict(n_trees=50, n_channels=20, C=0.01, bands=forest.BANDS, prior_weight=5)

model = MODEL.fit(full, **PARAMS)

## Экспорт

In [ ]:
folder = f"../submissions/{NAME}"
zip_path = save_submission(model, folder, full.ch_names)

## Проверка упаковки
Модель из ZIP загружается так же, как на платформе, и предсказывает test-окна по одному. Должно совпасть 100%.
(Это проверка упаковки, а не оценка качества: модель обучена в том числе на test.)

In [ ]:
packaged = load_submission(folder, full.ch_names)
same = (predict_streams(model, test) == predict_streams(packaged, test)).mean()
print(f"совпадение предсказаний: {same:.1%}")
assert same == 1.0

## Загрузка

Codabench → соревнование → **My Submissions** → выбрать активную фазу → загрузить ZIP из `submissions/` → дождаться **Finished**.
Если упало, смотреть первую ошибку в ingestion log.

Необязательно: официальная локальная проверка в их репозитории (нужен `benchopt` и подготовленные данные PROTEUS):

```bash
COMPET_SUBMISSION_DIR=/path/to/Brain-Decoding/submissions/forest \
    benchopt run tracks/bci_decoding -d "BCI[study=dreyer2026proteus]" \
    -s /path/to/Brain-Decoding/submissions/forest/submission.py
```